# LLM 직접 불러 보기 · OpenAI 형식

강의 12 ~ 17장에서 본 것을 실제 호출로 확인한다.

- 실행 환경: Jupyter + Deno 커널 (TypeScript). 준비 방법은 `README.md`
- 기본값은 이 PC 의 로컬 Ollama. OpenAI 로 바꾸려면 아래 첫 칸의 세 값만 바꾼다

In [ ]:
// 접속 정보: 이 세 값만 바꾸면 OpenAI, Ollama, vLLM 어디든 같은 코드로 돈다
const BASE_URL = Deno.env.get("LLM_BASE_URL") ?? "http://localhost:11434/v1";  // OpenAI: https://api.openai.com/v1
const API_KEY  = Deno.env.get("LLM_API_KEY")  ?? "ollama";                     // Ollama 는 아무 값이나 된다
const MODEL    = Deno.env.get("LLM_MODEL")    ?? "llama3.2";

## ① LLM 호출은 HTTP 요청 하나다

`messages` 배열을 담아 POST 한 번. 그게 전부다.

In [ ]:
const res = await fetch(`${BASE_URL}/chat/completions`, {
  method: "POST",
  headers: { "Content-Type": "application/json", Authorization: `Bearer ${API_KEY}` },
  body: JSON.stringify({
    model: MODEL,
    messages: [
      { role: "system", content: "너는 교육장 일정 담당이다." },
      { role: "user", content: "29일에 제2교육장 비어 있어?" },
    ],
  }),
});
const data = await res.json();
console.log(data.choices[0].message.content);

아무것도 조회하지 않았다. 모른다고 하거나, 그럴듯하게 지어낸다.

응답 전체를 보면 `choices[0].message` 에 답이, `usage` 에 토큰 수가 들어 있다.

In [ ]:
console.log(JSON.stringify(data, null, 2));

## ② LLM 은 지난 대화를 기억하지 않는다 (15장)

앞으로는 아래 `chat()` 으로 부른다. 하는 일은 위의 `fetch` 와 똑같다.

In [ ]:
type Message = { role: "system" | "user" | "assistant"; content: string };

async function chat(messages: Message[], extra: Record<string, unknown> = {}) {
  const res = await fetch(`${BASE_URL}/chat/completions`, {
    method: "POST",
    headers: { "Content-Type": "application/json", Authorization: `Bearer ${API_KEY}` },
    body: JSON.stringify({ model: MODEL, messages, ...extra }),
  });
  if (!res.ok) throw new Error(`${res.status} ${await res.text()}`);
  return await res.json();
}

두 번째 질문만 보내면, "그 시간" 이 언제인지 모른다.

In [ ]:
const noHistory = await chat([
  { role: "system", content: "너는 교육장 일정 담당이다." },
  { role: "user", content: "그럼 그 시간으로 잡아 줘" },
]);
console.log(noHistory.choices[0].message.content);

지난 대화를 같이 보내야 안다. 기억은 LLM 이 아니라 **내가 보내는 `messages` 배열**에 있다.

In [ ]:
const withHistory = await chat([
  { role: "system", content: "너는 교육장 일정 담당이다." },
  { role: "user", content: "29일에 제2교육장 비어 있어?" },
  { role: "assistant", content: "16시 이후가 비어 있습니다." },
  { role: "user", content: "그럼 그 시간으로 잡아 줘" },
]);
console.log(withHistory.choices[0].message.content);

## ③ 대화가 길어질수록 보내는 토큰이 쌓인다 (16 ~ 17장)

질문할 때마다 지난 대화를 통째로 다시 보낸다. 입력 토큰 수가 계속 늘어나는 걸 보자.

In [ ]:
const history: Message[] = [
  { role: "system", content: "너는 교육장 일정 담당이다. 한두 문장으로 짧게 답한다." },
];

async function ask(text: string) {
  history.push({ role: "user", content: text });
  const r = await chat(history);
  const reply = r.choices[0].message.content;
  history.push({ role: "assistant", content: reply });
  console.log(`보낸 메시지 ${history.length - 1}개 · 입력 토큰 ${r.usage.prompt_tokens}\n  → ${reply}\n`);
}

await ask("29일에 제2교육장 비어 있어?");
await ask("그럼 오후 4시로 잡아 줘");
await ask("참석자는 20명이야. 의자 충분해?");
await ask("지금까지 잡은 일정을 정리해 줘");

## ④ LLM 은 다음 토큰을 예측할 뿐이다 (12장)

`logprobs` 를 켜면 다음 토큰 후보와 확률을 받을 수 있다.

In [ ]:
let next;
for (let i = 0; i < 3; i++) {  // 로컬 서버는 가끔 logprobs 를 빼먹어서, 비어 있으면 다시 묻는다
  next = await chat(
    [{ role: "user", content: "다음 문장에 이어질 단어 하나만 써: 서울의 오늘 날씨는" }],
    { max_tokens: 1, logprobs: true, top_logprobs: 5 },
  );
  if (next.choices[0].logprobs) break;
}

const top = next.choices[0].logprobs?.content?.[0]?.top_logprobs;
if (!top) {
  console.log("이 서버나 모델은 logprobs 를 돌려주지 않았다. 고른 토큰:", next.choices[0].message.content);
} else {
  for (const c of top) {
    const p = Math.exp(c.logprob) * 100;
    console.log(`${JSON.stringify(c.token).padEnd(12)} ${"█".repeat(Math.round(p / 2)).padEnd(50)} ${p.toFixed(1)}%`);
  }
}

- 한국어는 한 글자가 여러 토큰으로 쪼개져서, 후보가 깨진 글자처럼 보일 수 있다
- `logprobs` 를 지원하지 않는 서버나 모델도 있다. OpenAI 추론 모델은 `max_tokens` 대신 `max_completion_tokens` 를 쓴다

## 정리

- LLM 호출은 `messages` 를 담은 요청 하나다
- 기억은 LLM 이 아니라 `messages` 배열에 있다. 이 배열을 만들고 관리하는 게 하네스의 일이다
- 다음: 여기에 도구를 붙인다